In [5]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,ImagePath,Label
0,1,images/img_000001.jpg,0
1,2,images/img_000002.jpg,1
2,3,images/img_000003.jpg,1
3,4,images/img_000004.jpg,1
4,5,images/img_000005.jpg,0


In [6]:
import torch

batch_size = 32
num_epochs = 10
learning_rate = 0.0001

device = torch.device("cuda")
device

device(type='cuda')

In [8]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2

class MyDataset(Dataset):
    def __init__(self, df, img_size=128, transform=None):
        self.df = df
        self.img_size = img_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.ImagePath.iloc[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        else:
            img = cv2.resize(img, (self.img_size, self.img_size))
            img = torch.tensor(img, dtype=torch.float32).permute(2,0,1)/255
        label = torch.tensor(self.df.Label.iloc[idx], dtype=torch.float32).view(1)
        return img, label

train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.Resize((128, 128)),
    T.ToTensor(),
])

valid_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 128)),
    T.ToTensor(),
])

from sklearn.model_selection import train_test_split
train_df, valid_df = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(train_df, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = MyDataset(valid_df, transform=valid_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [164]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,16,3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        
            nn.Conv2d(16,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        
            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        
            nn.Flatten(),
            nn.Linear(64*14*14, 128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.net(x)

model = Net().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
model

Net(
  (net): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=12544, out_features=128, bias=True)
    (11): ReLU()
    (12): Linear(in_features=128, out_features=64, bias=True)
    (13): ReLU()
    (14): Linear(in_features=64, out_features=32, bias=True)
    (15): ReLU()
    (16): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [165]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Training Loss: {total_loss / len(train_loader):.4f}")


def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    print(f'Test Loss: {test_loss / len(test_loader):.4f}')

In [166]:
for epoch in range(1, num_epochs+1):
    train(model, device, train_loader, criterion, optimizer, epoch)
    test(model, device, test_loader, criterion)

Epoch 1, Training Loss: 0.6923
Test Loss: 0.6951
Epoch 2, Training Loss: 0.6862
Test Loss: 0.6774
Epoch 3, Training Loss: 0.6708
Test Loss: 0.6562
Epoch 4, Training Loss: 0.6614
Test Loss: 0.6377
Epoch 5, Training Loss: 0.6395
Test Loss: 0.6175
Epoch 6, Training Loss: 0.6186
Test Loss: 0.5727
Epoch 7, Training Loss: 0.5974
Test Loss: 0.5755
Epoch 8, Training Loss: 0.5986
Test Loss: 0.7045
Epoch 9, Training Loss: 0.5851
Test Loss: 0.5695
Epoch 10, Training Loss: 0.5721
Test Loss: 0.5778


In [167]:
final_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128,128)),
    T.ToTensor(),
])

class FinalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.ImagePath.iloc[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        else:
            img = T.ToTensor()(img)
        return img

final_dataset = FinalDataset(df_test, transform=final_transform)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

In [168]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        results = torch.sigmoid(model(inputs)).cpu()
        preds.append(results)

preds = torch.cat(preds, dim=0).numpy().flatten()
preds = (preds >= 0.5).astype(int)
preds[:10]

array([0, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [169]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "Label": preds
})

df_preds.head()

,SampleID,Label
0,775,0
1,776,1
2,777,1
3,778,1
4,779,1


In [170]:
df_preds.to_csv("submission.csv", index=False)

In [171]:
preds

array([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0,
       1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0])